In [ ]:
for name in ["df_gold", "run_generalisierungstest", "results_generalisierung"]:
    print(name, "->", name in globals())

In [ ]:
%pip install xgboost
dbutils.library.restartPython()

In [ ]:
import pandas as pd
import numpy as np
import time
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

HORIZON = 3
N_TEST = 370

df_gold = spark.table("dev_workspace.p229.gold_df_features").toPandas()
df_gold = df_gold.sort_values("date").reset_index(drop=True)

def compute_da(y_true, y_pred):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    valid = y_true.notna() & y_pred.notna()
    return (y_true[valid] == y_pred[valid]).mean()

def run_generalisierungstest(target_price_col, df_gold=df_gold, horizon=HORIZON, n_test=N_TEST):
    df = df_gold.copy()

    future_price = df[target_price_col].shift(-horizon)
    new_target_col = f"target_up_{horizon}d_{target_price_col}"
    df[new_target_col] = (future_price > df[target_price_col]).astype("Int64")
    df.loc[future_price.isna(), new_target_col] = pd.NA

    n = len(df)
    test_start_idx = n - n_test

    exclude_cols = ["date", "target_up_1d", "target_up_3d", "target_up_5d", "target_up_10d",
                    "target_return_1d", "target_return_3d", "target_return_5d", "target_return_10d",
                    "target_action_3d", new_target_col]
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    X = df[feature_cols].values
    y = df[new_target_col].values

    print(f"--- Start: {target_price_col} ---")
    start_time = time.time()
    total_iters = n - test_start_idx

    preds = []
    for idx, i in enumerate(range(test_start_idx, n)):
        X_train, y_train = X[:i], y[:i]
        valid_train = ~pd.isna(y_train)
        X_train, y_train = X_train[valid_train], y_train[valid_train]

        model = XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42
        )
        model.fit(X_train, y_train)
        preds.append(model.predict(X[i:i+1])[0])

        if idx % 10 == 0:
            elapsed = time.time() - start_time
            print(f"  {idx}/{total_iters} ({elapsed:.0f}s vergangen)")

    y_test = y[test_start_idx:n]
    da = compute_da(y_test, np.array(preds))
    total_time = time.time() - start_time
    print(f"--- Fertig: {target_price_col} — DA: {da:.4%} — Dauer: {total_time:.0f}s ---\n")
    return da

# Sammelt alle Ergebnisse für die Zusammenfassung in Zelle 6
results_generalisierung = {"heizoel_sued (Original)": 0.6757}

In [ ]:
# Nur zum Testen der Geschwindigkeit — NICHT für die echte Auswertung verwenden
_ = run_generalisierungstest("heizoel_suedwest", n_test=20)

In [ ]:
results_generalisierung["heizoel_suedwest"] = run_generalisierungstest("heizoel_suedwest")

In [ ]:
results_generalisierung["heizoel_suedost"] = run_generalisierungstest("heizoel_suedost")

In [ ]:
results_generalisierung["heizoel_rheinmain"] = run_generalisierungstest("heizoel_rheinmain")

In [ ]:
results_generalisierung["diesel_sued"] = run_generalisierungstest("diesel_sued")

In [ ]:
summary = pd.DataFrame(
    [(k, v) for k, v in results_generalisierung.items()],
    columns=["Reihe", "DA"]
).sort_values("DA", ascending=False)
summary["DA"] = summary["DA"].apply(lambda x: f"{x:.2%}")
print(summary.to_string(index=False))

In [ ]:
diesel_cols = [c for c in df_gold.columns if "diesel" in c.lower()]
print(diesel_cols)

In [ ]:
results_generalisierung["diesel_suedwest"] = run_generalisierungstest("diesel_suedwest")

In [ ]:
results_generalisierung["diesel_rheinmain"] = run_generalisierungstest("diesel_rheinmain")

In [ ]:
summary = pd.DataFrame(
    [(k, v) for k, v in results_generalisierung.items()],
    columns=["Reihe", "DA"]
).sort_values("DA", ascending=False)
summary["DA"] = summary["DA"].apply(lambda x: f"{x:.2%}")
print(summary.to_string(index=False))